In [8]:
# ==========================================
# XGBoost Leak Detection Model Training & Fine-Tuning
# Hardware-Matched, Real-Time Ready
# ==========================================

import pandas as pd
import numpy as np
import os
import glob
import random
import matplotlib.pyplot as plt
import seaborn as sns
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report
)
import joblib
import warnings
warnings.filterwarnings('ignore')

# ==========================================
# CONFIGURATION
# ==========================================

DATA_PATH = '/kaggle/input/datasets/sarakhan24/trainingdata'
TUNING_DIR = "/kaggle/input/datasets/sarakhan24/tuningdataset"
OUTPUT_DIR = '/kaggle/working'
os.makedirs(OUTPUT_DIR, exist_ok=True)

RANDOM_SEED = 42
WINDOW_SIZE = 30  # 30-second rolling window for normalization

# Feature columns (Consistent with hardware and simulation)
FEATURE_COLS = [
    'F1_Lmin_Norm',
    'F2_Lmin_Norm',
    'Flow_Div_Norm',
    'Flow_Div_Trend',
    'P1_SPU_Norm',
    'P2_SPU_Norm',
    'Pres_Div_Norm',
    'Pres_Div_Trend',
]

COLUMN_NAMES = ['Timestamp', 'P1_SPU', 'F1_Lmin', 'F2_Lmin', 'P2_SPU', 'Label']

print("=" * 80)
print("XGBOOST LEAK DETECTION: TRAINING & FINE-TUNING")
print("=" * 80)

# ==========================================
# STEP 1: LOAD BASE TRAINING DATA
# ==========================================

print("\nSTEP 1: Loading Base Training Data")
csv_files = sorted(glob.glob(os.path.join(DATA_PATH, '*.csv')))
scenarios = []

for filepath in csv_files:
    df = pd.read_csv(filepath, header=None, skiprows=1, names=COLUMN_NAMES)
    for col in COLUMN_NAMES:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    scenarios.append(df.dropna())

print(f"✓ Loaded {len(scenarios)} base scenarios")

# ==========================================
# STEP 2: FEATURE ENGINEERING FUNCTION
# ==========================================

def create_features_rolling(df, window_size=30):
    df = df.copy()
    # Flow Normalization
    for col in ['F1_Lmin', 'F2_Lmin']:
        r_min = df[col].rolling(window=window_size, min_periods=1).min()
        r_max = df[col].rolling(window=window_size, min_periods=1).max()
        df[f'{col}_Norm'] = (df[col] - r_min) / (r_max - r_min + 1e-6)
    
    # Flow Divergence
    df['Flow_Div'] = df['F1_Lmin'] - df['F2_Lmin']
    df['Flow_Div_Norm'] = (df['Flow_Div'] - df['Flow_Div'].rolling(window_size).min()) / (df['Flow_Div'].rolling(window_size).max() - df['Flow_Div'].rolling(window_size).min() + 1e-6)
    df['Flow_Div_Trend'] = df['Flow_Div'].rolling(window_size).mean()

    # Pressure Normalization
    for col in ['P1_SPU', 'P2_SPU']:
        r_min = df[col].rolling(window=window_size, min_periods=1).min()
        r_max = df[col].rolling(window=window_size, min_periods=1).max()
        df[f'{col.replace("Pressure_", "Pres_")}_Norm'] = (df[col] - r_min) / (r_max - r_min + 1e-6)

    # Pressure Divergence
    df['Pres_Div'] = df['P2_SPU'] - df['P1_SPU']
    df['Pres_Div_Norm'] = (df['Pres_Div'] - df['Pres_Div'].rolling(window_size).min()) / (df['Pres_Div'].rolling(window_size).max() - df['Pres_Div'].rolling(window_size).min() + 1e-6)
    df['Pres_Div_Trend'] = df['Pres_Div'].rolling(window_size).mean()
    
    return df.fillna(0)

# Process base scenarios
scenarios_with_features = [create_features_rolling(df, window_size=WINDOW_SIZE) for df in scenarios]

# ==========================================
# STEP 3: DATA SPLITTING (BASE DATA)
# ==========================================

random.seed(RANDOM_SEED)
random.shuffle(scenarios_with_features)

train_count = int(len(scenarios_with_features) * 0.7)
val_count = int(len(scenarios_with_features) * 0.2)

df_train = pd.concat(scenarios_with_features[:train_count], ignore_index=True)
df_val = pd.concat(scenarios_with_features[train_count:train_count+val_count], ignore_index=True)
df_test = pd.concat(scenarios_with_features[train_count+val_count:], ignore_index=True)

X_train, y_train = df_train[FEATURE_COLS].values, df_train['Label'].values
X_val, y_val = df_val[FEATURE_COLS].values, df_val['Label'].values
X_test, y_test = df_test[FEATURE_COLS].values, df_test['Label'].values

# ==========================================
# STEP 4: INITIAL TRAINING
# ==========================================

print("\nSTEP 4: Initial Training on Base Dataset")
n_no_leak = (y_train == 0).sum()
n_leak = (y_train == 1).sum()
scale_pos_weight = n_no_leak / n_leak if n_leak > 0 else 1.0

xgb_params = {
    'n_estimators': 150,
    'max_depth': 5,
    'learning_rate': 0.1,
    'scale_pos_weight': scale_pos_weight,
    'eval_metric': 'logloss',
    'random_state': RANDOM_SEED,
    'tree_method': 'hist',
}

clf = XGBClassifier(**xgb_params)
clf.fit(X_train, y_train, eval_set=[(X_train, y_train), (X_val, y_val)], verbose=False)
print("✓ Base model training complete")

# ==========================================
# STEP 5: FINE-TUNING ON HARDWARE DATA
# ==========================================

print("\nSTEP 5: Fine-Tuning on Hardware Data (Nudging for Real-World Noise)")
hw_csv_files = [f for f in os.listdir(TUNING_DIR) if f.endswith('.csv')]

if not hw_csv_files:
    print("⚠️ No hardware CSV files found. Skipping fine-tuning.")
else:
    print(f"📂 Found {len(hw_csv_files)} hardware scenarios.")
    hw_df_list = [pd.read_csv(os.path.join(TUNING_DIR, f)) for f in hw_csv_files]
    df_hardware = pd.concat(hw_df_list, ignore_index=True)

    # Engineering features on hardware data
    df_hw_features = create_features_rolling(df_hardware, window_size=WINDOW_SIZE)
    X_finetune = df_hw_features[FEATURE_COLS].values
    y_finetune = df_hw_features['Label'].values

    # Adjust params for fine-tuning
    clf.set_params(learning_rate=0.005, n_estimators=100)
    
    # Continued training logic (Incremental Fit)
    clf.fit(X_finetune, y_finetune, xgb_model=clf.get_booster())
    print("✓ Fine-tuning complete")

# ==========================================
# STEP 6: EVALUATION (ON FINE-TUNED MODEL)
# ==========================================

print("\nSTEP 6: Detailed Evaluation (After Fine-Tuning)")

def evaluate_model(clf, X, y, set_name="Set"):
    y_pred = clf.predict(X)
    y_pred_proba = clf.predict_proba(X)[:, 1]
    
    acc = accuracy_score(y, y_pred)
    prec = precision_score(y, y_pred, zero_division=0)
    rec = recall_score(y, y_pred, zero_division=0)
    f1 = f1_score(y, y_pred, zero_division=0)
    
    print(f"\n{set_name} Metrics:")
    print(f"  Accuracy:  {acc:.4f} | Precision: {prec:.4f} | Recall: {rec:.4f} | F1: {f1:.4f}")
    return y_pred

# Detailed check on base test set and hardware tuning set
y_pred_test = evaluate_model(clf, X_test, y_test, "Base Test Set")
y_pred_hw = evaluate_model(clf, X_finetune, y_finetune, "Hardware Tuning Set")

# ==========================================
# STEP 7: SAVE FINE-TUNED MODEL
# ==========================================

model_path = os.path.join(OUTPUT_DIR, 'xgb_leak_detector_hardware_finetuned.json')
clf.save_model(model_path)
print(f"\n✓ Fine-tuned model saved: {model_path}")

XGBOOST LEAK DETECTION: TRAINING & FINE-TUNING

STEP 1: Loading Base Training Data
✓ Loaded 87 base scenarios

STEP 4: Initial Training on Base Dataset
✓ Base model training complete

STEP 5: Fine-Tuning on Hardware Data (Nudging for Real-World Noise)
📂 Found 28 hardware scenarios.
✓ Fine-tuning complete

STEP 6: Detailed Evaluation (After Fine-Tuning)

Base Test Set Metrics:
  Accuracy:  0.9535 | Precision: 0.9086 | Recall: 0.9526 | F1: 0.9300

Hardware Tuning Set Metrics:
  Accuracy:  0.8252 | Precision: 0.7909 | Recall: 0.8698 | F1: 0.8285

✓ Fine-tuned model saved: /kaggle/working/xgb_leak_detector_hardware_finetuned.json


In [7]:
import pandas as pd
import numpy as np
import os
import xgboost as xgb
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

# 1. Define Paths
# Update eval_dir to the folder containing your CSVs
eval_dir = "/kaggle/input/datasets/sarakhan24/evaldata/"
model_path = "/kaggle/working/xgb_leak_detector_hardware_finetuned.json"

# 2. Initialize and Load the Model
model = xgb.Booster()
model.load_model(model_path)
print("✅ Hardware-tuned model loaded successfully!")

# 3. Dynamically find all CSV files in the directory
hardware_files = [f for f in os.listdir(eval_dir) if f.endswith('.csv')]
hardware_files.sort()  # Ensures scenarios are processed in order

if not hardware_files:
    print(f"⚠️ No CSV files found in {eval_dir}")
else:
    print(f"📂 Found {len(hardware_files)} scenarios for evaluation.\n")

all_predictions = []
all_labels = []

# 4. Evaluation Loop
for file in hardware_files:
    file_path = os.path.join(eval_dir, file)
    df = pd.read_csv(file_path)
    
    # --- Feature Engineering ---
    # Ensure your create_features_rolling function is defined in your notebook
    features = create_features_rolling(df) 
    
    # Feature vector matching your ESP32 atomic packets
    feature_cols = ['F1_Lmin_Norm', 'F2_Lmin_Norm', 'Flow_Div_Norm', 
                    'Flow_Div_Trend', 'P1_SPU_Norm', 'P2_SPU_Norm',
                    'Pres_Div_Norm', 'Pres_Div_Trend']
    
    X = features[feature_cols].values
    y = df['Label'].values

    # XGBoost prediction
    dtest = xgb.DMatrix(X)
    y_prob = model.predict(dtest)
    
    # Use your optimized threshold (e.g., 0.4) instead of 0.5 if desired
    threshold = 0.5 
    y_pred = (y_prob > threshold).astype(int)
    
    all_predictions.extend(y_pred)
    all_labels.extend(y)
    
    # Per-scenario results
    print(f"📄 {file}:")
    print(f"   Accuracy:  {accuracy_score(y, y_pred):.4f}")
    print(f"   Precision: {precision_score(y, y_pred, zero_division=0):.4f}")
    print(f"   Recall:    {recall_score(y, y_pred, zero_division=0):.4f}")
    print(f"   F1-Score:  {f1_score(y, y_pred, zero_division=0):.4f}\n")

# 5. Final Overall Results
y_all = np.array(all_labels)
y_pred_all = np.array(all_predictions)

print(f"{'='*60}")
print("OVERALL HARDWARE PERFORMANCE SUMMARY")
print(f"{'='*60}")
print(f"Total Accuracy:  {accuracy_score(y_all, y_pred_all):.4f}")
print(f"Total Precision: {precision_score(y_all, y_pred_all, zero_division=0):.4f}")
print(f"Total Recall:    {recall_score(y_all, y_pred_all, zero_division=0):.4f}")
print(f"Total F1-Score:  {f1_score(y_all, y_pred_all, zero_division=0):.4f}")

print(f"\nOverall Confusion Matrix:")
print(confusion_matrix(y_all, y_pred_all))

print("\nDetailed Classification Report:")
print(classification_report(y_all, y_pred_all, target_names=['Normal', 'Leak']))

✅ Hardware-tuned model loaded successfully!
📂 Found 19 scenarios for evaluation.

📄 test_edge2.csv:
   Accuracy:  0.7551
   Precision: 0.5556
   Recall:    0.7143
   F1-Score:  0.6250

📄 test_edge3.csv:
   Accuracy:  0.8525
   Precision: 0.7955
   Recall:    1.0000
   F1-Score:  0.8861

📄 test_edge4.csv:
   Accuracy:  0.9624
   Precision: 0.9479
   Recall:    1.0000
   F1-Score:  0.9733

📄 test_large1.csv:
   Accuracy:  0.6066
   Precision: 0.5660
   Recall:    0.9677
   F1-Score:  0.7143

📄 test_large2.csv:
   Accuracy:  0.8833
   Precision: 0.8286
   Recall:    0.9667
   F1-Score:  0.8923

📄 test_large3.csv:
   Accuracy:  0.6410
   Precision: 0.8750
   Recall:    0.5385
   F1-Score:  0.6667

📄 test_large4.csv:
   Accuracy:  0.9000
   Precision: 0.8485
   Recall:    0.9655
   F1-Score:  0.9032

📄 test_medium1.csv:
   Accuracy:  0.9516
   Precision: 0.9062
   Recall:    1.0000
   F1-Score:  0.9508

📄 test_medium2.csv:
   Accuracy:  0.8136
   Precision: 0.9744
   Recall:    0.7917
   F1